In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the CSV files

In [2]:
btc = pd.read_csv("BTC_full_data.csv")
eth = pd.read_csv("ETH_full_data.csv")

btc.head()
eth.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Log_Return
0,2018-01-01,755.757019,782.530029,742.004028,772.640991,772.640991,2595760128,NaN
1,2018-01-02,772.346008,914.830017,772.346008,884.443970,884.443970,5783349760,0.135145
2,2018-01-03,886.000000,974.471008,868.450989,962.719971,962.719971,5093159936,0.084803
3,2018-01-04,961.713013,1045.079956,946.085999,980.921997,980.921997,6502859776,0.018730
4,2018-01-05,975.750000,1075.390015,956.325012,997.719971,997.719971,6683149824,0.016980


# 2. evaluation function

In [3]:
def evaluate_strategy(strategy_returns, risk_free_rate=0.03, trading_days=365):
    r = strategy_returns.dropna()

    cumulative = np.exp(r.cumsum())

    cumulative_pnl = cumulative.iloc[-1] - 1
    avg_daily_pnl = r.mean()
    daily_vol = r.std() 
    annual_return = r.mean() * trading_days
    annual_vol = r.std() * np.sqrt(trading_days)

    sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol != 0 else np.nan

    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()

    return {
        "Cumulative PnL": cumulative_pnl,
        "Average Daily PnL": avg_daily_pnl,
        "Daily Volatility": daily_vol, 
        "Annualised Return": annual_return,
        "Volatility": annual_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }

# 3. Define Bollinger Band (price level)
position = 1 → long when price is below lower band
position = -1 → short when price is above upper band
position = 0 → exit when price reverts to the rolling mean

In [4]:
def bollinger_price_csv(
    df,
    output_file,
    date_col="Date",
    price_col="Close",
    window=20,
    num_std=2
):

    df = df.copy()

    # format date
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    # price + log return
    df["price"] = df[price_col]
    df["log_return"] = np.log(df["price"] / df["price"].shift(1))
    df["log_return"] = df["log_return"].fillna(0)

    # Bollinger stats
    df["rolling_mean"] = df["price"].rolling(window).mean()
    df["rolling_std"] = df["price"].rolling(window).std()

    df["upper_band"] = df["rolling_mean"] + num_std * df["rolling_std"]
    df["lower_band"] = df["rolling_mean"] - num_std * df["rolling_std"]

    position = 0
    positions = []
    trades = []
    actions = []

    for i in range(len(df)):

        price = df.loc[i, "price"]
        mean = df.loc[i, "rolling_mean"]
        upper = df.loc[i, "upper_band"]
        lower = df.loc[i, "lower_band"]

        signal = 0
        action = "hold"

        if pd.isna(mean):
            position = 0

        elif position == 0:
            if price < lower:
                position = 1
                signal = 1
                action = "buy"

            elif price > upper:
                position = -1
                signal = -1
                action = "sell"

        elif position == 1:
            if price >= mean:
                position = 0
                signal = -1
                action = "sell"

        elif position == -1:
            if price <= mean:
                position = 0
                signal = 1
                action = "buy"

        positions.append(position)
        trades.append(signal)
        actions.append(action)

    df["trade"] = trades
    df["trade_action"] = actions
    df["position"] = positions
    out = df[[date_col, "price", "log_return", "trade", "trade_action", "position"]]
    out = out.rename(columns={date_col: "date"})

    # save csv
    out.to_csv(output_file, index=False)

    print(f"CSV exported → {output_file}")

    return out

# 4. Run strategy and create final metrics table

In [5]:
btc_bb = bollinger_price_csv(btc, "btc_bollinger_post_trade.csv")
eth_bb = bollinger_price_csv(eth, "eth_bollinger_post_trade.csv")

# Compute strategy returns: position is lagged by 1 to avoid look-ahead bias
btc_bb["strategy_return"] = btc_bb["position"].shift(1) * btc_bb["log_return"]
eth_bb["strategy_return"] = eth_bb["position"].shift(1) * eth_bb["log_return"]

# Evaluate both
btc_metrics = evaluate_strategy(btc_bb["strategy_return"])
eth_metrics = evaluate_strategy(eth_bb["strategy_return"])

# Post-trade tables: rows where a trade occurred
btc_trades = btc_bb[btc_bb["trade"] != 0].copy()
eth_trades = eth_bb[eth_bb["trade"] != 0].copy()

print("=== BTC Post-Trade Table ===")
print(btc_trades.to_string(index=False))
print(f"\nTotal BTC trades: {len(btc_trades)}")

print("\n=== ETH Post-Trade Table ===")
print(eth_trades.to_string(index=False))
print(f"\nTotal ETH trades: {len(eth_trades)}")

CSV exported → btc_bollinger_post_trade.csv
CSV exported → eth_bollinger_post_trade.csv
=== BTC Post-Trade Table ===
      date         price  log_return  trade trade_action  position  strategy_return
2018-02-02   8830.750000   -0.037756      1          buy         1        -0.000000
2018-02-14   9494.629883    0.099161     -1         sell         0         0.099161
2018-03-10   8866.000000   -0.051820      1          buy         1        -0.000000
2018-04-12   7889.250000    0.124127     -1         sell         0         0.124127
2018-04-15   8329.110352    0.042037     -1         sell        -1         0.000000
2018-05-10   9043.940430   -0.030623      1          buy         0         0.030623
2018-05-11   8441.490234   -0.068936      1          buy         1        -0.000000
2018-06-07   7678.240234    0.003165     -1         sell         0         0.003165
2018-06-10   6786.020020   -0.104293      1          buy         1        -0.000000
2018-07-02   6614.180176    0.035136     -1

# 5. Result table

In [6]:
results = pd.DataFrame({
    "Criteria": [
        "Cumulative pnl",
        "Average daily pnl",
        "Annualised return",
        "Max drawdown",
        "Daily volatility", 
        "Volatility",
        "Sharpe Ratio (risk-free rate assumed to be 3%)"
    ],
    "BTC": [
        btc_metrics["Cumulative PnL"],
        btc_metrics["Average Daily PnL"],
        btc_metrics["Annualised Return"],
        btc_metrics["Max Drawdown"],
        btc_metrics["Daily Volatility"],
        btc_metrics["Volatility"],
        btc_metrics["Sharpe Ratio"]
    ],
    "ETH": [
        eth_metrics["Cumulative PnL"],
        eth_metrics["Average Daily PnL"],
        eth_metrics["Annualised Return"],
        eth_metrics["Max Drawdown"],
        eth_metrics["Daily Volatility"],
        eth_metrics["Volatility"],
        eth_metrics["Sharpe Ratio"]
    ]
})

results

,Criteria,BTC,ETH
0,Cumulative pnl,-0.774877,-0.949464
1,Average daily pnl,-0.000511,-0.001022
2,Annualised return,-0.186389,-0.373134
3,Max drawdown,-0.925013,-0.970129
4,Daily volatility,0.026886,0.034582
5,Volatility,0.513651,0.660685
6,Sharpe Ratio (risk-free rate assumed to be 3%),-0.421276,-0.610176


The Bollinger Bands mean-reversion strategy generated persistently negative performance for both BTC and ETH. Although short-term price reversals occasionally occurred, the strategy was unable to offset the impact of prolonged trending market regimes. This resulted in substantial cumulative losses, large maximum drawdowns exceeding 90%, and negative Sharpe ratios, indicating poor risk-adjusted returns. The weaker performance observed for ETH relative to BTC is consistent with its higher volatility and stronger directional price movements. Overall, the findings suggest that a simple Bollinger-based mean-reversion approach is not well suited to highly volatile and trend-dominated cryptocurrency markets without additional trend-filtering or risk-management mechanisms.